In [1]:
import os
import tqdm
import numpy as np
import pandas as pd
from tushare import pro_api, set_token
from datetime import datetime, timedelta
from datatools import get_price
import time

In [2]:
set_token('b549c6e18f71105519447d872727cc2b7f0022f071c9eb27d0256ebb')
pro = pro_api()

### 交易日历

In [ ]:
#trade_cal = pro.trade_cal()
#trade_cal.to_csv('data/trade_cal.csv', index=False)
trade_cal = pd.read_csv('data/trade_cal.csv', dtype={'cal_date': str})
data_cal = trade_cal[(trade_cal['cal_date']>='20051230')&(trade_cal['cal_date']<'20260101')]

### 日线

In [7]:
data_path = 'data/daily_K/'
for _, d in tqdm.tqdm(data_cal.iterrows()):
    if d.is_open == 1:
        daily = pro.daily(trade_date=d['cal_date'])
        adj_factor = pro.adj_factor(trade_date=d['cal_date'])
        price = pd.merge(daily, adj_factor)
        price = price.drop(columns=['pre_close', 'change', 'pct_chg']).sort_values('ts_code').reset_index(drop=True)
        price['trade_date'] = pd.to_datetime(price['trade_date'])
        price.to_feather(os.path.join(data_path, f'stock-{d["cal_date"]}.ftr'))
    else:
        continue

2924it [11:48,  4.13it/s]


### 每日基础数据

In [8]:
data_path = 'data/daily_basic/'
for _, d in tqdm.tqdm(data_cal.iterrows()):
    if d.is_open == 1:
        daily_basic = pro.daily_basic(trade_date=d['cal_date']).sort_values('ts_code').reset_index(drop=True)
        daily_basic['trade_date'] = pd.to_datetime(daily_basic['trade_date'])
        daily_basic.to_feather(os.path.join(data_path, f'basic-{d["cal_date"]}.ftr'))
    else:
        continue

2924it [11:37,  4.19it/s]


In [ ]:
#industry = pro.index_member_all()
#industry.to_csv('data/industry.csv', index=False)
#allstock = pro.stock_basic()
#allstock.to_csv('data/allstock.csv', index=False)

### 股票列表

In [10]:
allstock = pd.read_csv('data/allstock.csv')
allstock_list = allstock.ts_code.tolist()
fins = []

In [14]:
len(fins)

3200

### 财务指标

In [ ]:
batch_size = 200
with tqdm.tqdm(total=len(allstock_list), desc="处理股票", unit="stock") as pbar:
    for i in range(0, len(allstock_list), batch_size):
        batch = allstock_list[i:i + batch_size]
        for d in batch:
            tmp = pro.fina_indicator(ts_code=d, start_date='20050930', end_date='20130929').drop_duplicates(subset=['ts_code', 'end_date'], keep='first')
            fins.append(tmp)
            pbar.update(1)
        
        # 每批次结束后休息
        if i + batch_size < len(allstock_list):
            time.sleep(35)

处理股票: 100%|██████████| 2286/2286 [11:46<00:00,  3.23stock/s]  


In [16]:
df_fins = pd.concat(fins)
for d in df_fins.end_date.unique():
    df_fins[df_fins['end_date'] == d].to_feather(os.path.join('data/finance/', f'fina-{d}.ftr'))

### 融资融券

In [30]:
data_path = 'data/rzrq/'
batch_size = 200
with tqdm.tqdm(total=len(data_cal), desc="处理股票", unit="stock") as pbar:
    for i in range(0, len(data_cal), batch_size):
        batch = data_cal[i:i + batch_size]
        for _, d in batch.iterrows():
            if d.is_open == 1:
                tmp = pro.margin_detail(trade_date=d['cal_date']).sort_values('ts_code').reset_index(drop=True)
                tmp['trade_date'] = pd.to_datetime(tmp['trade_date'])
                tmp.to_feather(os.path.join(data_path, f'margin-{d["cal_date"]}.ftr'))
                pbar.update(1)
            else:
                continue
        
        # 每批次结束后休息
        if i + batch_size < len(data_cal):
            time.sleep(30)

处理股票:  46%|████▋     | 3393/7307 [26:19<30:22,  2.15stock/s]   


Exception: 抱歉，您每分钟最多访问该接口200次，权限的具体详情访问：https://tushare.pro/document/1?doc_id=108。

### 资金流向&龙虎榜

In [ ]:
for _, d in tqdm.tqdm(data_cal.iterrows()):
    if d.is_open == 1:
        moneyflow = pro.moneyflow(trade_date=d['cal_date']).sort_values('ts_code').reset_index(drop=True)
        moneyflow['trade_date'] = pd.to_datetime(moneyflow['trade_date'])
        moneyflow.to_feather(os.path.join('data/moneyflow/', f'moneyflow-{d["cal_date"]}.ftr'))
        toplist = pro.top_list(trade_date=d['cal_date']).sort_values('ts_code').reset_index(drop=True)
        toplist['trade_date'] = pd.to_datetime(toplist['trade_date'])
        toplist.to_feather(os.path.join('data/toplist/', f'toplist-{d["cal_date"]}.ftr'))
    else:
        continue

3237it [19:55,  2.79it/s]

### 股东增减持

In [ ]:
# to do 改为按月
# #获取单日全部增减持数据
# for stock in tqdm.tqdm(allstock_list):
#     if len(df)<3000:
#         df.to_feather(os.path.join('data/holdertrade/', f'{stock}.ftr'))
#     else:
#         print(f'{stock} has too many records')

# batch_size = 100
# with tqdm.tqdm(total=len(allstock_list), desc="处理股票", unit="stock") as pbar:
#     for i in range(0, len(allstock_list), batch_size):
#         batch = allstock_list[i:i + batch_size]
#         for d in batch:
#             tmp = pro.stk_holdertrade(ts_code=stock, start_date='20060101', end_date='20251231')
#             fins.append(tmp)
#             pbar.update(1)
        
#         # 每批次结束后休息
#         if i + batch_size < len(allstock_list):
#             time.sleep(35)

  4%|▍         | 98/2286 [00:11<04:14,  8.59it/s]


Exception: 抱歉，您每分钟最多访问该接口100次，权限的具体详情访问：https://tushare.pro/document/1?doc_id=108。